# 12.8 二分搜尋法手刻演算法二：邊界二分搜尋（Lower / Upper Bound）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_12-8_binary_search_boundary_lower_upper_bound.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 Chapter 12.7 精確二分搜尋模型、`mid` 計算與雙指標收斂邏輯。

---

### 學習導覽：突破精確比對——邊界鎖定與重複元素的二分奧義

在上一小節（12.7 節）中，我們成功掌握了經典的「精確二分搜尋法」。只要目標在串列中，且元素各個不重複，演算法就能精準無誤地回傳它的索引。
然而，在真實世界與 APCS 競賽的嚴苛考驗中，我們面對的資料很少如此純淨：
- **重複元素的困惑**：如果串列中出現了一長串重複的數字 `[10, 20, 20, 20, 20, 30]`，當我們搜尋 20 時，上一節的精確二分搜尋會回傳哪一個？它可能命中第 2 個，也可能命中第 3 個，完全取決於長度切分！**它根本無法保證找出「第一個出現的 20」！**
- **範圍查詢的需求**：如果我們要找「第一個大於等於 60 分的及格門檻同學在哪裡？」，或者「成績落在 70 到 89 分之間的同學一共有幾個人？」，精確匹配直接宣告棄權！

為了解決這些痛點，高階演算法為我們引入了兩座精密的燈塔：
1. **下邊界（Lower Bound）**：尋找**第一個大於或等於（$\ge$）目標**的位置。
2. **上邊界（Upper Bound）**：尋找**第一個嚴格大於（$>$）目標**的位置。

本單元是通往下節 Python 內建 `bisect` 模組最關鍵的「底層過渡橋樑」。當你親手刻出這兩座邊界演算法，你將徹底通透二分搜尋的最高境界！

在本單元中，我們將透過 6 個平緩的微階梯，揭開邊界二分的奧秘：
1. **12.8.1 重複元素帶來的困惑**：多個相同數值時精確二分會回傳哪一個？
2. **12.8.2 下邊界搜尋（Lower Bound）**：尋找「第一個大於或等於 target」的位置。
3. **12.8.3 上邊界搜尋（Upper Bound）**：尋找「第一個嚴格大於 target」的位置。
4. **12.8.4 差一陷阱（Off-by-one Error）防禦**：閉區間 vs 左閉右開收斂。
5. **12.8.5 手刻下界搜尋模板演練**：重複元素中鎖定最左側出現位置。
6. **12.8.6 區間範圍統計**：利用下界與上界相減極速 $O(\log N)$ 統計出現次數。

讓我們握緊手術刀，進行微米級的二分邊界精準定位！

### 12.8.1 重複元素帶來的困惑：多個相同數值時精確二分會回傳哪一個？

#### 1. 生活故事比喻：走道上穿一模一樣校服的雙胞胎
想像你在學校司令台前，全校同學依照身高由矮到高排成一長縱隊。
隊伍中間剛好站了五位身高恰好都是 165 公分的同學（他們緊緊排在索引 10 到 14）。
如果教官大喊：「165 公分的同學舉手！」
上一節的精確二分搜尋演算法就像一位拿著大聲公的教官，他跑到隊伍正中間一問：「你 165 公分嗎？」
如果站在中間（例如索引 12）的同學說：「是！」
教官就立刻大喊：「找到了！在第 12 號！」然後吹哨收工！
但如果今天大隊接力教練走過來，下達了一道更精確的指令：
**「我要找的是 165 公分同學中，排在『最左邊第一個（最矮先報到）』的那位同學！」**
這時候，精確二分搜尋就傻眼了——因為它在第 12 號停下了腳步，他根本不知道第 10 號和第 11 號也是 165 公分！

#### 2. 底層運作機制：精確二分搜尋的隨機收斂本質
在精確二分搜尋中，只要 `arr[mid] == target`，程式碼就執行 `return mid` 提早結束。
對於數列 `[10, 20, 20, 20, 20, 30]`：
- 若目標是 `20`，第一次計算 `mid = (0 + 5) // 2 = 2`。
- `arr[2]` 剛好等於 `20`！程式碼立刻回傳索引 `2`！
然而，真正的「第一個 20」其實安靜地躺在索引 `1`！
精確二分搜尋回傳的位置，完全取決於串列總長度的奇偶切分，它可能回傳這群重複元素中的任何一個位置，具有不可預測的隨機性。

#### 3. 初學者常見陷阱：命中後改用線性往左倒退走
有些初學者發現這個問題後，寫出了這種混合體：
「用二分搜尋找到某個 20 之後，再用 `while` 迴圈一步一步往左倒退走，直到走到不是 20 為止。」
看似聰明，但如果數列中有整整 $100,000$ 個 20 呢？
二分搜尋花了 20 次比對找到中間，接著你的 `while` 迴圈往左走了 50,000 步！原本極速的 $O(\log N)$ 瞬間退化回龜速的 $O(N)$，徹底喪失了二分搜尋的意義！

#### 4. APCS 實戰視野
在 APCS 競賽中，資料包含大量重複鍵值是家常便飯。解決重複元素定位的唯一正道，是在二分搜尋的決策樹上動手術——即便命中目標，也不准停下，繼續向左半邊二分逼近！

In [ ]:
# 範例 12.8.1：精確二分在重複元素下的局限性實測
# 一個包含連續重複數值 50 的已排序串列
data = [10, 30, 50, 50, 50, 50, 50, 70, 90]
target = 50

print("資料串列：", data)
print("索引標註：", list(range(len(data))))
print("目標 50 在串列中跨越的索引範圍是: 2 到 6")

# 執行上一節的標準精確二分搜尋
def exact_binary_search(arr, x):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] == x:
            return m  # 命中即回傳
        elif arr[m] < x:
            l = m + 1
        else:
            r = m - 1
    return -1

hit_idx = exact_binary_search(data, target)
print(f"\n精確二分搜尋找到的索引是: {hit_idx}")
print(f"結論：它回傳了中間的索引 {hit_idx}，而不是最左側的第一個 50（索引 2）！")

In [ ]:
# 填空題 12.8.1：手動定位重複區間的首尾邊界
# 任務：觀察數列中數值 30 出現的「最左側索引」與「最右側索引」。
grades = [10, 20, 30, 30, 30, 40, 50]

# 請直接填入數值 30 第一次與最後一次出現的索引
first_pos = ___
last_pos = ___

print(f"數值 30 最左側出現在索引 {first_pos}，最右側出現在索引 {last_pos}")

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.1
# 任務說明：
# 給定已排序串列 nums = [2, 4, 4, 4, 4, 9, 15]。
# 請使用一個迴圈找出數值 4 在串列中出現的「第一個索引」與「最後一個索引」，
# 並印出格式化結果。
#
# 【公開測試資料 1】
# nums = [2, 4, 4, 4, 4, 9, 15]
# 預期輸出：
# 數值 4 首度出現： 1
# 數值 4 最後出現： 4
#
# 【公開測試資料 2】
# nums = [5, 5, 5]
# 預期輸出：
# 數值 5 首度出現： 0
# 數值 5 最後出現： 2
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
nums = [2, 4, 4, 4, 4, 9, 15]
first_idx = None
last_idx = None

for i, x in enumerate(nums):
    if x == 4:
        if first_idx is None:
            first_idx = i
        last_idx = i

print("數值 4 首度出現：", first_idx)
print("數值 4 最後出現：", last_idx)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.1
# 任務說明：
# 請設計一個微型展示：
# 給定一個包含 10 個元素的串列：arr = [1, 2, 2, 2, 2, 2, 2, 2, 2, 3]
# 搜尋目標 target = 2。
# 印出精確二分搜尋第一次比對時命中的索引，
# 證明它命中在最中央（索引 4），證明其無法保證找到最開頭（索引 1）。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
arr = [1, 2, 2, 2, 2, 2, 2, 2, 2, 3]
l, r = 0, len(arr) - 1
m = (l + r) // 2
print(f"初次計算 mid = {m}，arr[{m}] = {arr[m]}")
print(f"精確二分搜尋會直接回傳索引 {m}，但首個 2 其實在索引 1！")

### 12.8.2 下邊界搜尋（Lower Bound）：尋找「第一個大於或等於 target」的位置

#### 1. 生活故事比喻：及格門檻的守門員
想像大考中心放榜，考生的成績由低到高排好：`[45, 52, 58, 60, 60, 65, 78, 92]`。
及格標準是 60 分。教務處主任下達指令：
**「請找出全校『第一個及格（成績 $\ge 60$）』的同學站在哪裡？」**
如果有一群人剛好考 60 分，我們要找的是這群 60 分同學中**最左邊的第一位**！
如果全校根本沒有任何人剛好考 60 分呢？（例如成績是 `[45, 52, 58, 62, 70]`），主任要找的就是 58 分之後的下一位——**考 62 分的那位同學**！因為他是第一個跨過 60 分門檻的人！
這個數學定義——**「尋找數列中第一個滿足 $x \ge target$ 的元素索引」**，就是世界聞名的**下邊界（Lower Bound）**！

#### 2. 底層運作機制：命中也不停步的「向左擠壓」核心心法
如何讓二分搜尋精準鎖定「最左側的第一個 $\ge target$」？
核心心法只有一句話：**「只要當前數值大於等於目標（$arr[mid] \ge target$），代表答案可能是 mid，或者在 mid 的更左邊！因此我們將右邊界向左縮小（$right = mid - 1$），絕不停步，繼續向左逼近！」**
讓我們追蹤指針變化：
- 若 `arr[mid] < target`：這太小了，絕對不合格！答案保證在右邊：`left = mid + 1`。
- 若 `arr[mid] >= target`：這個合格！但我們想找「最左邊的第一個」，所以我們記錄可能的答案，並讓 `right = mid - 1` 繼續往左探測！
當迴圈結束時，**指標 `left` 剛好就會穩穩停在第一個大於等於目標的位置上！**

#### 3. 初學者常見陷阱：目標大於所有元素時的越界
若串列所有元素都小於目標（例如在 `[10, 20, 30]` 找 `100`）：
此時沒有任何人滿足 $\ge 100$。演算法執行完畢後，`left` 會一路右移，最終停在 `len(arr)`（即索引 3）處！
這代表：**若 `left == len(arr)`，代表全數列皆小於 target！** 呼叫端必須做好此邊界防護。

#### 4. APCS 實戰視野
Lower Bound 是 APCS 題目中最通用的搜尋武器！它不僅能解決重複元素定位，更能用於「找門檻」、「數值插入位置判定」以及「區間查詢」。掌握 Lower Bound，代表你已經學會了半個 `bisect` 模組！

In [ ]:
# 範例 12.8.2：手刻 Lower Bound（尋找第一個 >= target 的位置）
def lower_bound(arr, target):
    # 尋找第一個大於或等於 target 的元素索引
    # 若所有元素都小於 target，回傳 len(arr)
    left = 0
    right = len(arr) - 1
    ans = len(arr)  # 預設為長度（代表全小於 target）
    
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] >= target:
            ans = mid         # 找到一個合格候選，先記下來
            right = mid - 1   # 關鍵！絕不停步，繼續往左擠壓找更左側的！
        else:
            left = mid + 1    # 太小了，必須往右找
            
    return ans

# 測試 1：重複元素中鎖定「最左側第一個」
grades = [10, 30, 50, 50, 50, 50, 50, 70, 90]
idx_50 = lower_bound(grades, 50)
print("成績數列：", grades)
print(f"尋找第一個 >= 50 的位置: 索引 {idx_50} (數值: {grades[idx_50]})")
print(f"驗證：索引 {idx_50} 的前一位是 {grades[idx_50 - 1]}，成功鎖定最左側第一個 50！")

# 測試 2：目標不存在時，找出「第一個大於它的門檻值」
idx_60 = lower_bound(grades, 60)
print(f"\n尋找第一個 >= 60 的位置: 索引 {idx_60} (數值: {grades[idx_60]})")

In [ ]:
# 填空題 12.8.2：下邊界核心向左逼近條件
# 任務：在下邊界二分搜尋中，填入關鍵的 >= 判定與向左擠壓語句。
def find_lower_bound(arr, target):
    l = 0
    r = len(arr) - 1
    res = len(arr)
    while l <= r:
        m = (l + r) // 2
        # 合格條件：當前元素大於等於目標
        if arr[m] ___ target:
            res = m
            # 繼續向左尋找更前面的可能解
            r = m - ___
        else:
            l = m + 1
    return res

test_arr = [5, 12, 12, 12, 20]
print("第一個 >= 12 的位置索引：", find_lower_bound(test_arr, 12))

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.2
# 任務說明：
# 給定已排序數列 thresholds = [15, 20, 20, 20, 35, 50]。
# 請使用手刻 lower_bound 函數：
# 1. 查詢第一個 >= 20 的索引。
# 2. 查詢第一個 >= 30 的索引。
# 3. 查詢第一個 >= 100 的索引。
#
# 【公開測試資料 1】
# thresholds = [15, 20, 20, 20, 35, 50]
# 預期輸出：
# >= 20 索引： 1
# >= 30 索引： 4
# >= 100 索引： 6
#
# 【公開測試資料 2】
# thresholds = [10, 20]
# 預期輸出：
# >= 20 索引： 1
# >= 30 索引： 2
# >= 100 索引： 2
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def lower_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

thresholds = [15, 20, 20, 20, 35, 50]
print(">= 20 索引：", lower_bound(thresholds, 20))
print(">= 30 索引：", lower_bound(thresholds, 30))
print(">= 100 索引：", lower_bound(thresholds, 100))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.2
# 任務說明：
# 某遊戲角色升級所需經驗值表為已排序串列：exp_levels = [100, 250, 450, 700, 1000]
# （索引 0 代表升到 1 級需 100 exp，索引 1 代表升到 2 級需 250 exp...）
# 玩家目前累積了 500 exp。
# 請使用 lower_bound 找出「下一級升級門檻」所在的索引與所需總經驗值。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def lower_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

exp_levels = [100, 250, 450, 700, 1000]
cur_exp = 500
next_lvl_idx = lower_bound(exp_levels, cur_exp)
print(f"玩家目前 exp: {cur_exp}，下一個超越門檻位於索引 {next_lvl_idx}，需達到 {exp_levels[next_lvl_idx]} exp！")

### 12.8.3 上邊界搜尋（Upper Bound）：尋找「第一個嚴格大於 target」的位置

#### 1. 生活故事比喻：拍賣會上的嚴格加價者
想像在藝術品拍賣會上，拍賣官大喊：「目前最高出價是 50 萬元！請問有誰願意出『比 50 萬更高（嚴格大於 50 萬）』的價錢？」
如果台下有人舉牌出價 50 萬，拍賣官會搖搖頭：「50 萬跟我現在一樣，不算更高！」
拍賣官要找的是——全場出價清單中，**第一個出價達到 51 萬、55 萬（嚴格 $> 50$ 萬）的人！**
在數學上，這個操作就叫做**上邊界（Upper Bound）**：
**「在已排序數列中，尋找第一個嚴格大於（$> target$）的元素索引位置！」**

#### 2. 底層運作機制：從大於等於到嚴格大於的一字之差
請仔細觀察 Lower Bound 與 Upper Bound 的微小卻致命的差異：
- **Lower Bound（下邊界）**：尋找第一個 **$\ge target$** 的位置。
  - 條件是 `if arr[mid] >= target:`。
- **Upper Bound（上邊界）**：尋找第一個 **$> target$** 的位置。
  - 條件是 **`if arr[mid] > target:`**！
看到差別了嗎？
在 Upper Bound 中，即使 `arr[mid] == target`，因為它**沒有嚴格大於目標**，演算法會把它判定為「太小了/不夠大」，直接執行 `left = mid + 1`，跨越所有相等的元素，直奔右側第一個比它更大的數值！

#### 3. 初學者常見陷阱：搞混 Upper Bound 與「最後一個等於目標」
初學者常常誤以為 Upper Bound 回傳的是「最後一個等於目標的元素位置」。
請牢記：Upper Bound 回傳的是**「第一個『大於』目標的位置」**！
對於數列 `[10, 20, 20, 20, 30]`，搜尋 20：
- Lower Bound 回傳索引 `1`（第一個 20 的位置）。
- Upper Bound 回傳索引 `4`（數值 30，第一個嚴格大於 20 的位置）！
如果想知道「最後一個 20 在哪裡」，答案是 `upper_bound - 1`（索引 $4 - 1 = 3$）！

#### 4. APCS 實戰視野
Upper Bound 與 Lower Bound 就像一對孿生天使。兩者聯手，可以在 $O(\log N)$ 的神速下鎖定任何數值在數列中的「左邊界與右邊界」，這是統計區間個數不可或缺的雙重神技。

In [ ]:
# 範例 12.8.3：手刻 Upper Bound（尋找第一個 > target 的位置）
def upper_bound(arr, target):
    # 尋找第一個嚴格大於 target 的元素索引
    # 若所有元素都小於等於 target，回傳 len(arr)
    left = 0
    right = len(arr) - 1
    ans = len(arr)
    
    while left <= right:
        mid = (left + right) // 2
        # 關鍵差異：嚴格大於 >
        if arr[mid] > target:
            ans = mid         # 合格，先記下來
            right = mid - 1   # 繼續向左逼近找第一個大於的
        else:
            left = mid + 1    # 小於或等於 target，通通往右跨越！
            
    return ans

data = [10, 20, 20, 20, 30, 40]
target = 20
print("資料數列：", data)

u_idx = upper_bound(data, target)
print(f"尋找第一個 > {target} 的 Upper Bound 索引: {u_idx} (數值為: {data[u_idx]})")
print(f"最後一個等於 {target} 的元素位置為 upper_bound - 1 = {u_idx - 1} (數值: {data[u_idx - 1]})")

In [ ]:
# 填空題 12.8.3：Upper Bound 嚴格大於條件填空
# 任務：補齊 Upper Bound 的比較運算子。
def my_upper_bound(arr, x):
    l, r = 0, len(arr) - 1
    res = len(arr)
    while l <= r:
        m = (l + r) // 2
        # 請填入嚴格大於符號
        if arr[m] ___ x:
            res = m
            r = m - 1
        else:
            l = m + 1
    return res

vals = [10, 25, 25, 25, 50]
print("第一個嚴格大於 25 的位置索引：", my_upper_bound(vals, 25))

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.3
# 任務說明：
# 給定已排序數列 nums = [5, 10, 10, 10, 15, 20]。
# 請實作 upper_bound 函數：
# 1. 查詢第一個 > 10 的索引。
# 2. 查詢第一個 > 20 的索引。
# 3. 查詢第一個 > 0 的索引。
#
# 【公開測試資料 1】
# nums = [5, 10, 10, 10, 15, 20]
# 預期輸出：
# > 10 索引： 4
# > 20 索引： 6
# > 0 索引： 0
#
# 【公開測試資料 2】
# nums = [1, 2, 3]
# 預期輸出：
# > 10 索引： 3
# > 20 索引： 3
# > 0 索引： 0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def upper_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

nums = [5, 10, 10, 10, 15, 20]
print("> 10 索引：", upper_bound(nums, 10))
print("> 20 索引：", upper_bound(nums, 20))
print("> 0 索引：", upper_bound(nums, 0))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.3
# 任務說明：
# 某量產零件標準尺寸為 50 mm，允許生產序列中的零件存在誤差：
# parts = [48, 49, 50, 50, 50, 51, 52]
# 請使用 upper_bound 找出「第一個尺寸超標（嚴格大於 50 mm）」的零件位置與其尺寸。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def upper_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

parts = [48, 49, 50, 50, 50, 51, 52]
idx = upper_bound(parts, 50)
print(f"第一個超標零件位於索引 {idx}，實測尺寸為 {parts[idx]} mm！")

### 12.8.4 差一陷阱（Off-by-one Error）防禦：閉區間 `[left, right]` vs 左閉右開 `[left, right)` 邊界收斂

#### 1. 生活故事比喻：圍籬樁數量的千古難題
如果一條 100 公尺長的道路，每隔 10 公尺插一根木樁，請問總共需要幾根木樁？
初學者往往脫口而出：「10 根！」
但只要親自畫在紙上數一數，你會發現是 **11 根**（因為頭尾都要插樁）！
在電腦程式設計中，這種「差了 1」的微妙邊界錯誤，有一個專屬的名字叫做**差一錯誤（Off-by-one Error）**。
在二分搜尋中，差一錯誤是所有工程師的惡夢：
- 終止條件到底要寫 `<=` 還是 `<`？
- 右邊界到底要傳 `len(arr) - 1` 還是 `len(arr)`？
- 更新指標時到底要寫 `mid` 還是 `mid - 1`？
如果把兩套不同的區間哲學混在一起，程式不是漏算最後一個元素，就是直接掉進無窮迴圈！

#### 2. 底層運作機制：兩大流派的嚴格對照表
二分搜尋在學術界有兩套截然不同但各自自洽的規範體系：

| 規格維度 | **雙閉區間流派 `[left, right]`（強烈推薦）** | **左閉右開流派 `[left, right)`** |
| :--- | :--- | :--- |
| **右指針初始值** | `right = len(arr) - 1` | `right = len(arr)` |
| **while 條件** | `while left <= right:`（有等號） | `while left < right:`（無等號） |
| **太小往右走** | `left = mid + 1` | `left = mid + 1` |
| **太大往左走** | `right = mid - 1` | `right = mid` |
| **收斂退出狀態** | `left == right + 1` | `left == right` |

#### 3. 初學者常見陷阱：張飛打岳飛的語法大亂鬥
最慘痛的 Bug 就是把兩派混著寫：例如右指標初始化寫了 `right = len(arr)`，但迴圈條件卻寫了 `while left <= right:`！
這會導致當搜尋極端數值時，`mid` 計算出 `len(arr)`，當場觸發 `IndexError: list index out of range` 崩潰！

#### 4. APCS 實戰視野
**本教材強烈建議初學者終身貫徹「雙閉區間 `[left, right]`」！**
因為它的語意最直觀：左指針指向第一個有效位置 `0`，右指針指向最後一個有效位置 `len - 1`，兩邊都是合法的真實下標，絕無越界或遺漏之慮。

In [ ]:
# 範例 12.8.4：雙閉區間流派防禦規範演示
# 堅定遵守雙閉區間四大金剛：
# 1. right = len(arr) - 1
# 2. while left <= right:
# 3. left = mid + 1
# 4. right = mid - 1

arr = [10, 20, 30, 40, 50]
target = 50  # 測試最右端邊界

l = 0
r = len(arr) - 1  # 閉區間右端點：4

found_idx = -1
while l <= r:
    m = (l + r) // 2
    print(f"掃描中: [{l}, {r}], mid={m}, val={arr[m]}")
    if arr[m] == target:
        found_idx = m
        break
    elif arr[m] < target:
        l = m + 1
    else:
        r = m - 1

print(f"邊界命中結果: 索引 {found_idx}，完全無任何 Off-by-one 偏差！")

In [ ]:
# 填空題 12.8.4：雙閉區間規範嚴格檢查
# 任務：補齊雙閉區間二分搜尋的右邊界初始化與 while 條件。
items = [1, 3, 5, 7, 9]

left = 0
# 閉區間右界初始化：長度減 1
right = len(items) - ___

# 閉區間迴圈條件：必須有小於等於
while left ___ right:
    mid = (left + right) // 2
    if items[mid] == 7:
        print("命中 7！")
        break
    elif items[mid] < 7:
        left = mid + 1
    else:
        right = mid - 1

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.4
# 任務說明：
# 給定單一元素串列 single = [99]。
# 請使用雙閉區間標準寫法，驗證搜尋 99 與 100 均能正常運作且不崩潰。
# 輸出兩者的搜尋結果索引（不存在輸出 -1）。
#
# 【公開測試資料 1】
# single = [99]
# 預期輸出：
# 搜尋 99 結果： 0
# 搜尋 100 結果： -1
#
# 【公開測試資料 2】
# single = [5]
# 預期輸出：
# 搜尋 99 結果： -1
# 搜尋 100 結果： -1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
single = [99]

def search_single(arr, target):
    l, r = 0, len(arr) - 1
    while l <= r:
        m = (l + r) // 2
        if arr[m] == target:
            return m
        elif arr[m] < target:
            l = m + 1
        else:
            r = m - 1
    return -1

print("搜尋 99 結果：", search_single(single, 99))
print("搜尋 100 結果：", search_single(single, 100))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.4
# 任務說明：
# 請設計一個微型測試程式，專門用來檢驗空串列空邊界：
# 當傳入空串列 arr = [] 時，二分搜尋會發生什麼事？
# 驗證在雙閉區間下：left = 0, right = -1，while 0 <= -1 條件直接為 False 退出，
# 完全不會進入迴圈，安全回傳 -1，展現零長度防禦的完美境界。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def safe_bsearch(arr, target):
    l = 0
    r = len(arr) - 1  # 若空串列，r == -1
    while l <= r:     # 0 <= -1 直接為 False！
        m = (l + r) // 2
        if arr[m] == target:
            return m
        elif arr[m] < target:
            l = m + 1
        else:
            r = m - 1
    return -1

empty_list = []
print("空串列搜尋結果：", safe_bsearch(empty_list, 10))
print("完美安全通過，無任何 IndexError 異常！")

### 12.8.5 手刻下界搜尋模板演練：重複元素中鎖定最左側出現位置

#### 1. 生活故事比喻：全班最早交卷的滿分同學
想像期末大考考場，全校前十名榜單公布：`[80, 85, 90, 100, 100, 100, 100]`。
校長走過來，手裡拿著一張榮譽狀，大聲說：「請考 100 分同學中，『最早交卷登記的那位第一號狀元』上台領獎！」
這時候，我們絕對不能隨便挑一個 100 分的同學充數。
我們要的是——**在重複的 100 分群體中，精準鎖定最左邊（索引最小）的那個人**！
利用我們在 12.8.2 學到的 Lower Bound 模板，我們現在要把它打造成一個乾淨俐落、能夠精準判斷「目標到底在不在、若在則回傳最左側下標」的頂級實戰函數！

#### 2. 底層運作機制：Lower Bound + 存在性驗證
Lower Bound 會回傳第一個 $\ge target$ 的索引 `idx`。
此時我們只需進行兩步安全驗證：
1. **是否越界**：檢查 `idx < len(arr)`（確認目標沒有大於全體元素）。
2. **數值是否剛好相等**：檢查 `arr[idx] == target`！
- 若兩個條件皆成立：太棒了！**`idx` 就是該數值在串列中「第一次（最左側）出現的真實下標」**！
- 若任一條件不成立：證明串列中根本不存在任何等於 target 的元素！

#### 3. 初學者常見陷阱：直接把 Lower Bound 的回傳值當成存在
請切記：Lower Bound 找到的是「大於等於」！
如果在 `[10, 20, 30]` 找 `25`，Lower Bound 會回傳索引 `2`（數值 30）！
如果你沒有檢查 `arr[idx] == 25`，你就會把 30 誤當成 25，引發嚴重邏輯錯誤！

#### 4. APCS 實戰視野
這套「Lower Bound 尋找下界 + 存在性驗證」的組合，是 APCS 實作三級與競賽選手處理重複資料的標配解法。它兼具二分搜尋的對數速度與最左側邊界定位的精準性。

In [ ]:
# 範例 12.8.5：重複元素最左側索引精準定位函數
def find_first_occurrence(arr, target):
    # 精確尋找 target 在已排序串列中「第一次出現（最左側）」的索引
    # 若 target 不在串列中，回傳 -1
    l, r = 0, len(arr) - 1
    lower_idx = len(arr)
    
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            lower_idx = m
            r = m - 1  # 命中或更大，通通向左逼近
        else:
            l = m + 1
            
    # 兩步安全檢驗：
    # 1. 索引未越界
    # 2. 該位置數值恰好等於 target
    if lower_idx < len(arr) and arr[lower_idx] == target:
        return lower_idx
    return -1

# 實測驗證
data = [10, 20, 30, 30, 30, 30, 30, 40, 50]
print("測試資料：", data)

target_val = 30
first_idx = find_first_occurrence(data, target_val)
print(f"數值 {target_val} 第一次出現的最左側索引為: {first_idx}")
print(f"驗證：data[{first_idx}] = {data[first_idx]}，且其前一位是 {data[first_idx - 1]}")

# 測試不存在數值
missing = 35
print(f"搜尋不存在數值 {missing}: {find_first_occurrence(data, missing)}")

In [ ]:
# 填空題 12.8.5：最左側定位存在性檢驗填空
# 任務：在取得 lower_idx 後，填寫雙重驗證條件。
def first_match(arr, x):
    l, r = 0, len(arr) - 1
    pos = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= x:
            pos = m
            r = m - 1
        else:
            l = m + 1
            
    # 請填入未越界（pos < len(arr)）與相等條件
    if pos < len(arr) and arr[pos] ___ x:
        return pos
    return -1

scores = [50, 60, 60, 60, 70]
print("第一個 60 的位置：", first_match(scores, 60))

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.5
# 任務說明：
# 給定已排序數列 nums = [1, 2, 2, 2, 3, 4, 4, 5]。
# 請呼叫 find_first_occurrence 函數：
# 1. 查詢數值 2 的首度出現索引。
# 2. 查詢數值 4 的首度出現索引。
# 3. 查詢數值 9 的首度出現索引（不存在回傳 -1）。
#
# 【公開測試資料 1】
# nums = [1, 2, 2, 2, 3, 4, 4, 5]
# 預期輸出：
# 2 首度出現： 1
# 4 首度出現： 5
# 9 首度出現： -1
#
# 【公開測試資料 2】
# nums = [10, 10, 10]
# 預期輸出：
# 2 首度出現： -1
# 4 首度出現： -1
# 9 首度出現： -1
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def find_first_occurrence(arr, target):
    l, r = 0, len(arr) - 1
    lower_idx = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            lower_idx = m
            r = m - 1
        else:
            l = m + 1
    if lower_idx < len(arr) and arr[lower_idx] == target:
        return lower_idx
    return -1

nums = [1, 2, 2, 2, 3, 4, 4, 5]
print("2 首度出現：", find_first_occurrence(nums, 2))
print("4 首度出現：", find_first_occurrence(nums, 4))
print("9 首度出現：", find_first_occurrence(nums, 9))

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.5
# 任務說明：
# 請設計一個微型展示：
# 實作 find_last_occurrence(arr, target) 函數，
# 利用 Upper Bound 的概念（尋找第一個 > target 的位置再減 1），
# 精準找出 target 在數列中「最後一次（最右側）」出現的索引位置，若無回傳 -1。
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def find_last_occurrence(arr, target):
    l, r = 0, len(arr) - 1
    upper_idx = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            upper_idx = m
            r = m - 1
        else:
            l = m + 1
    last_idx = upper_idx - 1
    if 0 <= last_idx < len(arr) and arr[last_idx] == target:
        return last_idx
    return -1

data = [5, 8, 8, 8, 8, 12]
print(f"數列: {data}")
print("數值 8 最後一次出現的最右側索引為:", find_last_occurrence(data, 8))

### 12.8.6 區間範圍統計：利用下界與上界相減，極速 $O(\log N)$ 統計某數值出現的總次數

#### 1. 生活故事比喻：火車站月台的區間乘車人數計數
想像台鐵火車站月台上站滿了依車廂號碼整齊排隊的旅客：`[1車, 2車, 2車, 2車, 2車, 3車]`。
站長拿著無線電問廣播室：「請問第 2 節車廂的排隊旅客一共有幾個人？」
查驗員不需要一個一個點頭數人頭。
他只需要做兩件事：
1. 找出第 2 車廂旅客隊列的「起點刻度（Lower Bound）」：第 1 號旅客。
2. 找出第 3 車廂（第一個嚴格大於 2 車）隊列的「起點刻度（Upper Bound）」：第 5 號旅客。
站長拿起粉筆算一下：$5 - 1 = 4$！
**「答案是 4 個人！」**
這就是資訊科學中最令人拍案叫絕的數學奇蹟——**以邊界相減極速計數！**

#### 2. 底層運作機制：$[LowerBound, UpperBound)$ 的左閉右開區間長度
在已排序數列中，任何特定數值 $X$ 所佔據的空間，剛好就是：
- 起點：$LowerBound(X)$（第一個等於 $X$ 的位置）。
- 終點：$UpperBound(X)$（第一個大於 $X$ 的位置，不包含在 $X$ 的群體內）。
這構成了一個完美的數學左閉右開區間 $[L, R)$！
在集合論與數線上，左閉右開區間內的整數個數公式就是：
$$Count(X) = UpperBound(X) - LowerBound(X)$$
無論這個數值在數列中重複出現了 10 次、1,000 次、還是 1,000,000 次：
我們**完全不需要寫任何 `for` 迴圈計數**！
只需進行兩次二分搜尋（分別花費 $O(\log N)$），將兩個下標輕輕一減，**以 $O(\log N)$ 的神速瞬間算出總個數！**

#### 3. 初學者常見陷阱：當目標不存在時的減法結果
如果目標在串列中根本不存在（例如在 `[10, 30]` 找 `20`）：
- $LowerBound(20)$ 會找到 30 的位置（索引 1）。
- $UpperBound(20)$ 也會找到 30 的位置（索引 1）。
兩者相減：$1 - 1 = 0$！
看見了嗎？**公式依然完全成立！** 當目標不存在時，兩界相減自動精確回傳 `0`！絕不需要額外的 `if-else` 分支！

#### 4. APCS 實戰視野
「給定 $Q$ 次詢問，每次詢問某數字在海量數列中出現的頻率」是 APCS 實作第三級的常客。若每次查詢都用 `count()` 進行線性走訪，總時間為 $Q \times N$（超時掛掉）；若改用「兩次手刻邊界二分相減」，時間瞬間縮減為 $Q \times \log N$，輕鬆斬獲 100 分滿分！

In [ ]:
# 範例 12.8.6：Upper Bound - Lower Bound 秒殺元素個數統計
# 封裝好的雙邊界工具
def lower_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

def upper_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

# 測試大量重複數列
records = [10, 20, 20, 20, 20, 20, 20, 35, 50]
print("已排序紀錄：", records)

target = 20
lb = lower_bound(records, target)
ub = upper_bound(records, target)
count = ub - lb

print(f"\n查詢數值 {target}:")
print(f"  Lower Bound 索引 = {lb}")
print(f"  Upper Bound 索引 = {ub}")
print(f"  計算出現總次數: {ub} - {lb} = {count} 次！")
assert count == records.count(target), "統計結果不符！"

# 測試不存在數值
missing = 25
lb_m = lower_bound(records, missing)
ub_m = upper_bound(records, missing)
print(f"\n查詢不存在數值 {missing}:")
print(f"  Lower Bound = {lb_m}, Upper Bound = {ub_m}")
print(f"  計算出現總次數: {ub_m} - {lb_m} = {ub_m - lb_m} 次！(完美吻合)")

In [ ]:
# 填空題 12.8.6：極速區間個數計算
# 任務：利用 upper_bound 與 lower_bound 計算指定分數的人數。
scores = [60, 70, 70, 70, 70, 85, 90]
target_score = 70

# 取得下界與上界
lb = lower_bound(scores, target_score)
ub = upper_bound(scores, target_score)

# 請填入個數計算公式：上界減下界
freq = ___ - ___

print(f"獲得 {target_score} 分的同學總人數為：", freq)

In [ ]:
# ==========================================
# [4] Code 練習題 12.8.6
# 任務說明：
# 給定已排序數列 grades = [50, 60, 60, 60, 75, 80, 80, 90]。
# 請撰寫程式：
# 使用 upper_bound(grades, target) - lower_bound(grades, target) 公式，
# 依序計算 [60, 80, 100] 的出現次數並輸出清單。
#
# 【公開測試資料 1】
# targets = [60, 80, 100]
# 預期輸出：
# 各目標出現次數： [3, 2, 0]
#
# 【公開測試資料 2】
# targets = [50, 90]
# 預期輸出：
# 各目標出現次數： [1, 1]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：
def lower_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

def upper_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

grades = [50, 60, 60, 60, 75, 80, 80, 90]
targets = [60, 80, 100]

ans_counts = [upper_bound(grades, t) - lower_bound(grades, t) for t in targets]
print("各目標出現次數：", ans_counts)

In [ ]:
# ==========================================
# [5] Code 挑戰題 12.8.6
# 任務說明：
# 某統計題目給定已排序的身高數據（公分）：
# heights = [155, 160, 162, 165, 168, 170, 172, 175, 180, 185]
# 請利用 lower_bound 與 upper_bound，
# 一行不跑全盤迴圈，以 O(log N) 極速計算「身高落在 [165, 175] 閉區間之內」的同學一共有幾位？
# （提示：等價於 upper_bound(175) - lower_bound(165)）
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：
def lower_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] >= target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

def upper_bound(arr, target):
    l, r = 0, len(arr) - 1
    ans = len(arr)
    while l <= r:
        m = (l + r) // 2
        if arr[m] > target:
            ans = m
            r = m - 1
        else:
            l = m + 1
    return ans

heights = [155, 160, 162, 165, 168, 170, 172, 175, 180, 185]
L_bound = lower_bound(heights, 165)  # 第一個 >= 165
R_bound = upper_bound(heights, 175)  # 第一個 > 175
in_range_count = R_bound - L_bound

print(f"身高在 [165, 175] 區間內的同學共有: {in_range_count} 位")
print(f"符合名單區間子串列: {heights[L_bound:R_bound]}")

### 學習總結與通關回顧

恭喜你順利通關 **12.8 二分搜尋法手刻演算法二：邊界二分搜尋（Lower / Upper Bound）**！

在本單元中，你攻克了二分搜尋體系最精密、最考驗功力的邊界難關：
- **重複元素定位瓶頸**：
  - 精確二分搜尋命中即回傳，無法保證最左側或最右側邊界。
- **下邊界（Lower Bound）核心契約**：
  - 尋找「第一個 $\ge target$」的位置。
  - 判定條件：`arr[mid] >= target`，命中後 `right = mid - 1` 繼續向左逼近。
- **上邊界（Upper Bound）核心契約**：
  - 尋找「第一個 $> target$」的位置。
  - 判定條件：`arr[mid] > target`，嚴格大於時才向左收縮。
- **雙閉區間防禦四鐵律**：
  - `right = len - 1`、`while left <= right:`、`left = mid + 1`、`right = mid - 1`，徹底杜絕差一錯誤（Off-by-one Error）。
- **$O(\log N)$ 區間個數秒殺公式**：
  - 元素總出現次數 $= UpperBound(X) - LowerBound(X)$。
  - 範圍計數 $[A, B]$ 總數 $= UpperBound(B) - LowerBound(A)$。

---
**下一關預告**：手刻邊界二分固然神勇，但每次都要寫 20 行是不是有點累？Python 標準庫早就把這套心法用純 C 語言封裝好了！下一節 **12.9 內建二分搜尋模組：bisect 與數值區間查詢應用** 將帶你無縫對接標準庫，秒殺數值級距與動態有序序列！